# 04 - Qwen2.5-VL-3B on POPE (3,000) and AMBER (3,000) (Google Colab, GPU)

Runs the same protocol as notebooks 01 and 03 (prompt `"{question} Answer yes or no."`, last prompt token, every layer, forced yes/no scoring, image-level 70/15/15 split) for `qwen25vl-3b`, on POPE (the same seeded 3,000-question subset as the other models) and AMBER (1,000 queries per subset). Each dataset goes through a smoke test; the full run only starts if `answer_mass` is close to 1.

1. Use a GPU runtime (T4 or better) and keep bfloat16.
2. Set `USE_DRIVE = True` so outputs survive disconnects. Budget several hours per dataset on a T4; use `DATASETS = ["pope"]`, then `["amber"]` to run one per session.
3. After a disconnect, re-run all cells with the same config; finished shards are skipped.
4. Outputs: `pope_qwen25vl-3b_{features.pt, metadata.parquet, run_config.json}` and the same for `amber`. Do not commit the `.pt` files.
5. Copy the files into `results/cached_features/`, then run the evaluation (printed by the last cell). The verification printout should show 0 missing, 0 duplicates, 0 non-finite values and no `!!! FLAG` line.

## Config

In [ ]:
MODEL_KEY = "qwen25vl-3b"
DATASETS = ["pope", "amber"]    # run order
POPE_SUBSET_SIZE = 3000         # same seeded subset as the other models
AMBER_PER_SUBSET = 1000         # random queries per AMBER subset
SMOKE_N = 20                    # questions in the smoke test
MIN_ANSWER_MASS = 0.9           # minimum mean answer_mass to start the full run
LAYERS = "all"
REPO_URL = "https://github.com/RUANLASS/SC4001Assignment_2026.git"
BRANCH = "main"
USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/sc4001_extra"
AMBER_HF_IMAGES = None          # fallback image source, e.g. "visual-preference/AMBER"

## a. Setup

In [ ]:
import os, re, sys, subprocess
from pathlib import Path
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "nvidia-smi not found")
if not Path("src/extract_features_multi.py").exists():
    target = Path("/content/SC4001Assignment_2026")
    if not target.exists():
        r = subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(target)], capture_output=True, text=True)
        print(r.stdout, r.stderr)
    assert (target / "src" / "extract_features_multi.py").exists(), "clone failed (private repo? wrong BRANCH?): upload the repo folder instead"
    os.chdir(target)
print("working dir:", os.getcwd())
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-extra.txt"])
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU, then re-run"
OUT_DIR = "results/cached_features"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive"); OUT_DIR = DRIVE_DIR; os.makedirs(OUT_DIR, exist_ok=True)
print("outputs ->", OUT_DIR)
print(subprocess.run([sys.executable, "src/extract_features_multi.py", "--list-models"], capture_output=True, text=True).stdout)

## b. AMBER data (only if "amber" is in DATASETS)
Metadata and ground truth come from the AMBER GitHub repo, pixels from the Google-Drive zip linked in its README (or from the Hugging Face mirror if `AMBER_HF_IMAGES` is set).

In [ ]:
if "amber" in DATASETS:
    if not Path("data/amber_repo").exists():
        subprocess.check_call(["git", "clone", "-q", "https://github.com/junyangwang0410/AMBER", "data/amber_repo"])
    have_images = Path("data/amber_images").exists() and any(Path("data/amber_images").rglob("AMBER_*.jpg"))
    if AMBER_HF_IMAGES is None and not have_images:
        import gdown
        gdown.download(id="1MaCHgtupcZUjf007anNl4_MV0o4DjXvl", output="data/amber_images.zip", quiet=False)
        subprocess.check_call(["unzip", "-q", "-o", "data/amber_images.zip", "-d", "data/amber_images"])
AMBER_ARGS = ["--amber-hf-images", AMBER_HF_IMAGES] if AMBER_HF_IMAGES else []

## c. Smoke test with gate, then full run
For each dataset: (1) smoke test on `SMOKE_N` questions (nothing is written); (2) stop if the mean `answer_mass` is below `MIN_ANSWER_MASS`; (3) otherwise start the resumable full run (checkpoint every 100 questions).

In [ ]:
def run_cmd(args):
    """Run a command, echo its output live, return (exit code, full output)."""
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        print(line, end=""); lines.append(line)
    p.wait()
    return p.returncode, "".join(lines)

def common_args(ds):
    a = ["--model", MODEL_KEY, "--dataset", ds, "--layers", LAYERS, "--out-dir", OUT_DIR, "--per-subset", str(AMBER_PER_SUBSET)]
    if ds == "pope":
        a += ["--subset-size", str(POPE_SUBSET_SIZE)]
    return a + (AMBER_ARGS if ds == "amber" else [])

for ds in DATASETS:
    print(f"\n{'=' * 30} {MODEL_KEY} on {ds}: smoke test {'=' * 30}")
    code, out = run_cmd([sys.executable, "src/extract_features_multi.py", *common_args(ds), "--smoke", str(SMOKE_N)])
    m = re.search(r"mean answer_mass ([0-9.]+)", out)
    assert code == 0 and m, "smoke test failed (see output above); do not start the full run"
    mass = float(m.group(1))
    assert mass >= MIN_ANSWER_MASS, f"answer_mass {mass:.3f} < {MIN_ANSWER_MASS}: STOP. The model is not answering yes/no; check the yes/no token ids and prompt above."
    print(f"\nanswer_mass {mass:.3f} OK -> starting the full {ds} run")
    code, out = run_cmd([sys.executable, "src/extract_features_multi.py", *common_args(ds)])
    assert code == 0, "full run failed (see output above); re-run this cell to resume"
    assert "!!! FLAG" not in out, "verification raised a FLAG (see output above); do not use these files until it is understood"

## d. Output files

In [ ]:
print(subprocess.run(["ls", "-la", OUT_DIR], capture_output=True, text=True).stdout)
print("Copy the *_qwen25vl-3b_* files into your local results/cached_features/ (never commit the .pt files), then on your laptop run e.g.:")
print("  python src/probe_transfer_eval.py --models qwen2vl-2b smolvlm-500m qwen25vl-3b --probe logreg --out-dir results/final/cross_model_logreg")